In [1]:
import torch
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
import pandas as pd
import numpy as np
import re
import os
import json
import zipfile

# Проверяем версии
import transformers
import accelerate
print(f"Transformers версия: {transformers.__version__}")
print(f"Accelerate версия: {accelerate.__version__}")
print(f"PyTorch версия: {torch.__version__}")

# Проверяем доступность GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используем устройство: {device}")

Transformers версия: 5.0.0
Accelerate версия: 1.12.0
PyTorch версия: 2.10.0+cu128
Используем устройство: cuda


# 1. СКАЧИВАНИЕ И РАСПАКОВКА

In [2]:
print("\nСкачиваем архив с датасетом...")
!wget -q --show-progress https://storage.yandexcloud.net/ai-2025/gazeta.zip

if os.path.exists('gazeta.zip'):
    file_size = os.path.getsize('gazeta.zip')
    print(f"Архив скачан, размер: {file_size/1024/1024:.2f} MB")

print("\nРаспаковываем архив...")
with zipfile.ZipFile('gazeta.zip', 'r') as zip_ref:
    zip_ref.extractall('./gazeta_data/')



Скачиваем архив с датасетом...
gazeta.zip          100%[===================>] 151.74M  23.5MB/s    in 7.8s    
Архив скачан, размер: 151.74 MB

Распаковываем архив...


# 2. ЧТЕНИЕ JSONL ФАЙЛОВ

In [3]:
print("\nЧитаем JSONL файлы...")

def read_jsonl_to_list(filename):
    """Читает JSONL файл и возвращает список текстов"""
    texts = []
    filepath = f'./gazeta_data/{filename}'

    with open(filepath, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f):
            try:
                json_obj = json.loads(line.strip())
                if 'text' in json_obj and json_obj['text']:
                    texts.append(json_obj['text'])
            except:
                continue

    print(f"  {filename}: {len(texts)} примеров")
    return texts

# Читаем данные
train_texts = read_jsonl_to_list('gazeta_train.jsonl')
val_texts = read_jsonl_to_list('gazeta_val.jsonl')
test_texts = read_jsonl_to_list('gazeta_test.jsonl')

print(f"\nИтого:")
print(f"  Train: {len(train_texts)} примеров")
print(f"  Validation: {len(val_texts)} примеров")
print(f"  Test: {len(test_texts)} примеров")

# Показываем пример
if train_texts:
    print(f"\nПример текста:")
    print(train_texts[0][:300] + "...")


Читаем JSONL файлы...
  gazeta_train.jsonl: 52400 примеров
  gazeta_val.jsonl: 5265 примеров
  gazeta_test.jsonl: 5770 примеров

Итого:
  Train: 52400 примеров
  Validation: 5265 примеров
  Test: 5770 примеров

Пример текста:
«По итогам 2011 года чистый отток может составить примерно $80 млрд, в следующем году — около $20 млрд. При этом мы ожидаем, что со второго полугодия 2012 года начнется приток капитала», — заявил «Интерфаксу» замминистра экономического развития Андрей Клепач. Официальные прогнозы по выводу капитала ...


# 3. ПОДГОТОВКА ДАННЫХ

In [4]:
def clean_text(text):
    """Очистка текста"""
    if text is None:
        return None
    text = re.sub(r'\s+', ' ', str(text)).strip()
    if text and text[-1] not in ['.', '!', '?']:
        text += '.'
    return text

print("\nОчищаем тексты...")

# Очищаем тексты (берем подмножество для скорости)
train_texts = [clean_text(t) for t in train_texts[:3000] if clean_text(t) and len(clean_text(t)) > 50]
val_texts = [clean_text(t) for t in val_texts[:500] if clean_text(t) and len(clean_text(t)) > 50]

print(f"После очистки:")
print(f"  Train: {len(train_texts)} примеров")
print(f"  Validation: {len(val_texts)} примеров")


Очищаем тексты...
После очистки:
  Train: 3000 примеров
  Validation: 500 примеров


# 4. ЗАГРУЗКА МОДЕЛИ И ТОКЕНИЗАТОРА

In [5]:
model_name = "ai-forever/rugpt3medium_based_on_gpt2"
print(f"\nЗагружаем модель: {model_name}")

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# Устанавливаем pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Перемещаем модель на устройство
model.to(device)
print("Модель загружена успешно!")


Загружаем модель: ai-forever/rugpt3medium_based_on_gpt2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/1.73G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.73G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3medium_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель загружена успешно!


# 5. ТОКЕНИЗАЦИЯ ДАННЫХ

In [6]:
print("\nТокенизируем данные...")

def tokenize_texts(texts, max_length=128):
    """Токенизация списка текстов"""
    return tokenizer(
        texts,
        truncation=True,
        padding=False,
        max_length=max_length,
        return_tensors=None
    )

# Токенизируем
train_encodings = tokenize_texts(train_texts)
val_encodings = tokenize_texts(val_texts)

# Создаем Dataset объекты
train_dataset = Dataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask']
})

val_dataset = Dataset.from_dict({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask']
})

print(f"Размер train dataset: {len(train_dataset)}")
print(f"Размер val dataset: {len(val_dataset)}")



Токенизируем данные...
Размер train dataset: 3000
Размер val dataset: 500


6. НАСТРОЙКА ОБУЧЕНИЯ

In [9]:
print("\nНастраиваем обучение...")

# Data collator для языкового моделирования
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Для GPT нужно False
    pad_to_multiple_of=8,
)

# Параметры обучения
training_args = TrainingArguments(
    output_dir="./rugpt3-fake-news",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

# Создаем Trainer (без параметра tokenizer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    # tokenizer убран отсюда
)

print("Trainer создан успешно!")



`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



Настраиваем обучение...
Trainer создан успешно!


7. ОБУЧЕНИЕ

In [10]:
print("\nНачинаем обучение...")
trainer.train()

# Сохраняем модель
model.save_pretrained("./rugpt3-fake-news-final")
tokenizer.save_pretrained("./rugpt3-fake-news-final")
print("Модель сохранена в ./rugpt3-fake-news-final")


Начинаем обучение...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.076681,3.002480
2,2.327498,3.087637
3,1.929462,3.274290


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель сохранена в ./rugpt3-fake-news-final


8. ФУНКЦИЯ ГЕНЕРАЦИИ

In [11]:
def generate_news(prompt, max_length=150, temperature=0.8):
    """Генерирует новость с контролем окончания"""

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=100,
            min_length=30,
            do_sample=True,
            temperature=temperature,
            top_k=50,
            top_p=0.9,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    new_text = generated[len(prompt):].strip()

    # Обрезаем до последнего знака препинания
    match = re.search(r'.*[.!?]', new_text)
    if match:
        new_text = match.group(0)
    elif new_text and new_text[-1] not in ['.', '!', '?']:
        new_text += '.'

    return prompt + " " + new_text

9. ГЕНЕРАЦИЯ 5+ ПРИМЕРОВ

In [12]:
print("\n" + "="*60)
print("СГЕНЕРИРОВАННЫЕ ФЕЙКОВЫЕ НОВОСТИ")
print("="*60)

prompts = [
    "Сенсация! Учёные из Новосибирска обнаружили, что",
    "Официальное заявление: Правительство РФ приняло решение",
    "Шокирующие новости: в Москве сегодня",
    "Международное агентство сообщает, что",
    "В ходе эксперимента исследователи выяснили, что"
]

generated_news = []
for i, prompt in enumerate(prompts):
    print(f"\nНовость {i+1}:")
    print(f"Промпт: {prompt}")

    news = generate_news(prompt, temperature=0.8)
    generated_news.append(news)
    print(f"Результат: {news}")
    print("-" * 50)


СГЕНЕРИРОВАННЫЕ ФЕЙКОВЫЕ НОВОСТИ

Новость 1:
Промпт: Сенсация! Учёные из Новосибирска обнаружили, что
Результат: Сенсация! Учёные из Новосибирска обнаружили, что в ДНК человека содержится генетический код двух других инопланетных рас. «В геноме нашего народа и на самом деле есть две совершенно различные формы жизни — те же существа, которые были обнаружены ранее, но их оказалось гораздо больше», — сказал сотрудник Центра космических исследований РАН Александр Гроттер. В частности, у них обнаружились более древние гены: одни связаны с умением летать и плавать, а другие отвечают за способность к выживанию.
--------------------------------------------------

Новость 2:
Промпт: Официальное заявление: Правительство РФ приняло решение
Результат: Официальное заявление: Правительство РФ приняло решение о приостановлении действия соглашения между Украиной и Россией по транзиту газа в Европу через территорию Украины. Как отмечается, это касается транзитного коридора «Северный поток — 2», которы

10. ПРОВЕРКА ОКОНЧАНИЙ (для 4 баллов)

In [13]:
print("\n" + "="*60)
print("ПРОВЕРКА ОКОНЧАНИЙ ПРЕДЛОЖЕНИЙ")
print("="*60)

all_good = True
for i, news in enumerate(generated_news):
    last_char = news[-1]
    if last_char in ['.', '!', '?']:
        status = "✓"
    else:
        status = "✗"
        all_good = False
    print(f"Новость {i+1}: последний символ '{last_char}' - {status}")

if all_good:
    print("\n Все новости заканчиваются корректно!")
else:
    print("\n Некоторые новости обрываются - можно настроить параметры генерации")

# Сохраняем результаты
with open("generated_fake_news.txt", "w", encoding="utf-8") as f:
    for i, news in enumerate(generated_news):
        f.write(f"Новость {i+1}:\n{news}\n\n")

print("\n Готово! Результаты сохранены в 'generated_fake_news.txt'")



ПРОВЕРКА ОКОНЧАНИЙ ПРЕДЛОЖЕНИЙ
Новость 1: последний символ '.' - ✓
Новость 2: последний символ '.' - ✓
Новость 3: последний символ '.' - ✓
Новость 4: последний символ '.' - ✓
Новость 5: последний символ '.' - ✓

 Все новости заканчиваются корректно!

 Готово! Результаты сохранены в 'generated_fake_news.txt'
